# Eksperimen Heart Disease Classification
Notebook ini dibuat untuk memenuhi **Kriteria 1 - Experimentation** pada Submission Akhir kelas **Membangun Sistem Machine Learning** di Dicoding.

### Tahapan Eksperimen:
1. **Data Loading**: Membaca dataset, cek shape, dan tipe data.
2. **EDA**: Analisis missing values, duplikasi, outlier, distribusi, korelasi, dan analisis target.
3. **Data Preprocessing**: Handling missing values, duplikasi, encoding, feature engineering, scaling, dan train-test split.
4. **Menyimpan Dataset**: Menyimpan dataset preprocessed ke folder `dataset_preprocessed/`.

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style untuk visualisasi
sns.set_theme(style="whitegrid")

In [ ]:
# Load dataset
data_path = "dataset/heart_disease.csv"
df = pd.read_csv(data_path)

# Cek shape dataset
print(f"Shape dataset: {df.shape[0]} baris, {df.shape[1]} kolom")

In [ ]:
# Cek tipe data setiap kolom
print("Tipe data setiap kolom:")
print(df.dtypes)

In [ ]:
# Tampilkan 5 baris pertama
df.head()

## 2. Exploratory Data Analysis (EDA)

### A. Missing Values

In [ ]:
# Cek missing values
print("Jumlah missing values per kolom:")
print(df.isnull().sum())

### B. Duplicate Values

In [ ]:
# Cek duplicate values
duplicates_count = df.duplicated().sum()
print(f"Jumlah baris duplikat: {duplicates_count}")

### C. Outlier Analysis

In [ ]:
# Analisis Outlier menggunakan Boxplot untuk variabel numerik kontinu
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

plt.figure(figsize=(14, 8))
for i, col in enumerate(continuous_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot of {col}')
plt.tight_layout()
plt.show()

### D. Distribution Analysis

In [ ]:
# Distribusi variabel numerik
plt.figure(figsize=(14, 8))
for i, col in enumerate(continuous_cols, 1):
    plt.subplot(2, 3, i)
    sns.histplot(df[col], kde=True, bins=20)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### E. Correlation Analysis

In [ ]:
# Analisis Korelasi antar fitur
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix of Features')
plt.show()

### F. Target Analysis

In [ ]:
# Distribusi target
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df)
plt.title('Target Column Distribution (0 = Normal, 1 = Heart Disease)')
plt.show()

print("Frekuensi Target:")
print(df['target'].value_counts())
print("\nPersentase Target:")
print(df['target'].value_counts(normalize=True) * 100)

## 3. Data Preprocessing

In [ ]:
# Handling Duplicates
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Shape setelah membuang duplikat: {df_clean.shape}")

In [ ]:
# Handling Missing Values (Imputasi menggunakan median)
for col in ['chol', 'trestbps']:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

print("Missing values setelah imputasi:")
print(df_clean.isnull().sum())

In [ ]:
# Feature Engineering
df_feat = df_clean.copy()

# 1. Risk Ratio: cholesterol / resting blood pressure
df_feat['chol_bps_ratio'] = df_feat['chol'] / df_feat['trestbps']

# 2. Age group categorical feature (discretization)
# 0: Young (<45), 1: Middle-aged (45-60), 2: Elderly (>60)
df_feat['age_group'] = pd.cut(df_feat['age'], bins=[0, 45, 60, np.inf], labels=[0, 1, 2]).astype(int)

# 3. Heart rate / age ratio
df_feat['hr_age_ratio'] = df_feat['thalach'] / df_feat['age']

df_feat.head()

In [ ]:
# Encoding Categorical Features & Scaling
X = df_feat.drop(columns=['target'])
y = df_feat['target']

# Identifikasi kolom multi-category untuk One-Hot Encoding
multi_cat_cols = ['cp', 'restecg', 'slope', 'thal', 'age_group']
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'chol_bps_ratio', 'hr_age_ratio']

# One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=multi_cat_cols, drop_first=True)
dummy_cols = [col for col in X_encoded.columns if any(mc in col for mc in multi_cat_cols)]
X_encoded[dummy_cols] = X_encoded[dummy_cols].astype(int)

X_encoded.head()

In [ ]:
# Train Test Split (80% Train, 20% Test)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling numeric features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

X_test_scaled = X_test.copy()
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Gabungkan kembali fitur dan target untuk disimpan
train_preprocessed = X_train_scaled.copy()
train_preprocessed['target'] = y_train

test_preprocessed = X_test_scaled.copy()
test_preprocessed['target'] = y_test

print(f"Train shape: {train_preprocessed.shape}")
print(f"Test shape: {test_preprocessed.shape}")

## 4. Simpan Dataset Hasil Preprocessing

In [ ]:
# Menyimpan dataset hasil preprocessing ke folder dataset_preprocessed/
output_dir = "dataset_preprocessed"
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(output_dir, "train.csv")
test_path = os.path.join(output_dir, "test.csv")

train_preprocessed.to_csv(train_path, index=False)
test_preprocessed.to_csv(test_path, index=False)

print(f"Preprocessed train dataset saved successfully to {train_path}")
print(f"Preprocessed test dataset saved successfully to {test_path}")